In [0]:
# AI-Powered Security Anomaly Detection on Azure
#
# Author: Pooja Borade
#
# Technologies:
# - Python
# - Databricks
# - Pandas
# - NumPy
# - Scikit-learn
# - Azure OpenAI
# - Power BI
#
# Objective:
# Detect unusual login events using Machine Learning
# (Isolation Forest) and explain suspicious events
# using Azure OpenAI.

print("AI Security Anomaly Detection Project Started")

AI Security Anomaly Detection Project Started


In [0]:
# Import Required Libraries


import pandas as pd
import numpy as np

from datetime import datetime, timedelta
from datetime import timedelta

print("Libraries imported successfully!")

Libraries imported successfully!


In [0]:
# Step 1: Define Users and Their Normal Login Locations


# Make random numbers repeatable
np.random.seed(42)

# Users in our organization
users = ["Alice", "Bob", "Charlie", "David", "Eva"]

# Normal country for each user
normal_country = {
    "Alice": "US",
    "Bob": "US",
    "Charlie": "UK",
    "David": "US",
    "Eva": "Canada"
}

# Azure resources users can access
resources = [
    "VM-1",
    "Storage-1",
    "SQL-DB-1",
    "KeyVault-1",
    "WebApp-1"
]

print("Users Loaded Successfully")
print(users)

Users Loaded Successfully
['Alice', 'Bob', 'Charlie', 'David', 'Eva']


In [0]:
# Step 2: Generate Normal Login Activity


rows = []

start_date = datetime(2026, 7, 1)

for i in range(500):

    # Random employee
    user = np.random.choice(users)

    # Business hours only (8 AM - 6 PM)
    hour = np.random.randint(8, 18)

    # Random day in July
    timestamp = start_date + timedelta(
        days=np.random.randint(0, 30),
        hours=hour,
        minutes=np.random.randint(0, 60)
    )

    rows.append({
        "User": user,
        "Timestamp": timestamp,
        "Country": normal_country[user],
        "Hour": hour,
        "Status": "Success",
        "Resource": np.random.choice(resources)
    })

print(f"Generated {len(rows)} normal login events")

Generated 500 normal login events


In [0]:
# Step 3: Create Pandas DataFrame


df = pd.DataFrame(rows)
original_df = df.copy()

print("Dataset Created Successfully!")
print("Number of Records:", len(df))

Dataset Created Successfully!
Number of Records: 520


In [0]:
# Step 4: Display First Five Records

df.head()

,User,Timestamp,Country,Hour,Status,Resource
0,David,2026-07-29 15:20:00,US,15,Success,Storage-1
1,Charlie,2026-07-11 14:10:00,UK,14,Success,WebApp-1
2,David,2026-07-24 15:02:00,US,15,Success,WebApp-1
3,Bob,2026-07-12 15:29:00,US,15,Success,Storage-1
4,David,2026-07-01 12:11:00,US,12,Success,Storage-1


In [0]:
# Step 5: Understand the Dataset

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   User       500 non-null    object        
 1   Timestamp  500 non-null    datetime64[ns]
 2   Country    500 non-null    object        
 3   Hour       500 non-null    int64         
 4   Status     500 non-null    object        
 5   Resource   500 non-null    object        
dtypes: datetime64[ns](1), int64(1), object(4)
memory usage: 23.6+ KB


In [0]:
# Step 6: Summary Statistics

df.describe(include="all")

,User,Timestamp,Country,Hour,Status,Resource
count,500,500,500,500.000000,500,500
unique,5,NaN,3,NaN,1,5
top,David,NaN,US,NaN,Success,Storage-1
freq,115,NaN,303,NaN,500,116
mean,NaN,2026-07-16 05:32:02.400000,NaN,12.624000,NaN,NaN
min,NaN,2026-07-01 08:46:00,NaN,8.000000,NaN,NaN
25%,NaN,2026-07-08 11:53:45,NaN,10.000000,NaN,NaN
50%,NaN,2026-07-16 09:36:00,NaN,13.000000,NaN,NaN
75%,NaN,2026-07-24 10:15:15,NaN,15.000000,NaN,NaN
max,NaN,2026-07-30 17:49:00,NaN,17.000000,NaN,NaN


In [0]:
# Step 7: Inject Suspicious Login Events

anomaly_countries = ["Russia", "China", "North Korea", "Iran"]

for i in range(20):

    user = np.random.choice(users)

    timestamp = start_date + timedelta(
        days=np.random.randint(0, 30),
        hours=np.random.randint(0, 5),      # Midnight to 4 AM
        minutes=np.random.randint(0, 60)
    )

    rows.append({
        "User": user,
        "Timestamp": timestamp,
        "Country": np.random.choice(anomaly_countries),
        "Hour": timestamp.hour,
        "Status": "Failed",
        "Resource": np.random.choice(resources)
    })

print("20 suspicious login events added.")

20 suspicious login events added.


In [0]:
# Step 8: Refresh DataFrame


df = pd.DataFrame(rows)

print("Dataset Updated Successfully")
print("Total Records:", len(df))

Dataset Updated Successfully
Total Records: 520


In [0]:
# Step 9: View Last Records


df.tail(20)

,User,Timestamp,Country,Hour,Status,Resource
500,Alice,2026-07-03 04:08:00,Iran,4,Failed,SQL-DB-1
501,David,2026-07-01 03:03:00,North Korea,3,Failed,VM-1
502,Eva,2026-07-07 03:51:00,Russia,3,Failed,KeyVault-1
503,David,2026-07-28 01:36:00,Iran,1,Failed,WebApp-1
504,Charlie,2026-07-07 02:27:00,China,2,Failed,WebApp-1
505,Eva,2026-07-22 04:54:00,China,4,Failed,SQL-DB-1
506,Eva,2026-07-22 04:09:00,North Korea,4,Failed,Storage-1
507,Eva,2026-07-01 04:20:00,China,4,Failed,Storage-1
508,Eva,2026-07-21 04:17:00,Russia,4,Failed,WebApp-1
509,David,2026-07-17 02:48:00,Iran,2,Failed,SQL-DB-1


In [0]:
# Step 10: Convert Categorical Data to Numbers

from sklearn.preprocessing import LabelEncoder

# Create LabelEncoder object
encoder = LabelEncoder()

# Convert text columns into numeric values
df["User"] = encoder.fit_transform(df["User"])
df["Country"] = encoder.fit_transform(df["Country"])
df["Status"] = encoder.fit_transform(df["Status"])
df["Resource"] = encoder.fit_transform(df["Resource"])

print("Categorical columns encoded successfully!")

df.head()

Categorical columns encoded successfully!


,User,Timestamp,Country,Hour,Status,Resource
0,3,2026-07-29 15:20:00,6,15,1,2
1,2,2026-07-11 14:10:00,5,14,1,4
2,3,2026-07-24 15:02:00,6,15,1,4
3,1,2026-07-12 15:29:00,6,15,1,2
4,3,2026-07-01 12:11:00,6,12,1,2


In [0]:
# Step 11: Select Features


features = [
    "User",
    "Country",
    "Hour",
    "Status",
    "Resource"
]

X = df[features]

print("Feature Matrix Created")
print(X.head())

Feature Matrix Created
   User  Country  Hour  Status  Resource
0     3        6    15       1         2
1     2        5    14       1         4
2     3        6    15       1         4
3     1        6    15       1         2
4     3        6    12       1         2


In [0]:
# Step 12: Train Isolation Forest Model


from sklearn.ensemble import IsolationForest

# Create the model
model = IsolationForest(
    n_estimators=100,
    contamination=0.04,
    random_state=42
)

# Train the model
model.fit(X)

print("Isolation Forest model trained successfully!")

Isolation Forest model trained successfully!


In [0]:
# Step 13: Predict Anomalies


df["Prediction"] = model.predict(X)

print("Predictions completed!")

df.head()

Predictions completed!


,User,Timestamp,Country,Hour,Status,Resource,Prediction
0,3,2026-07-29 15:20:00,6,15,1,2,1
1,2,2026-07-11 14:10:00,5,14,1,4,1
2,3,2026-07-24 15:02:00,6,15,1,4,1
3,1,2026-07-12 15:29:00,6,15,1,2,1
4,3,2026-07-01 12:11:00,6,12,1,2,1


In [0]:
# Step 14: Count Normal vs Anomalies


print(df["Prediction"].value_counts())

Prediction
 1    499
-1     21
Name: count, dtype: int64


In [0]:
# Step 15: Display Suspicious Login Events


anomalies = df[df["Prediction"] == -1]

print("Number of anomalies detected:", len(anomalies))

anomalies.head(10)

Number of anomalies detected: 21


,User,Timestamp,Country,Hour,Status,Resource,Prediction
430,4,2026-07-12 08:57:00,0,8,1,4,-1
500,0,2026-07-03 04:08:00,2,4,0,1,-1
501,3,2026-07-01 03:03:00,3,3,0,3,-1
502,4,2026-07-07 03:51:00,4,3,0,0,-1
503,3,2026-07-28 01:36:00,2,1,0,4,-1
504,2,2026-07-07 02:27:00,1,2,0,4,-1
505,4,2026-07-22 04:54:00,1,4,0,1,-1
506,4,2026-07-22 04:09:00,3,4,0,2,-1
507,4,2026-07-01 04:20:00,1,4,0,2,-1
508,4,2026-07-21 04:17:00,4,4,0,4,-1


In [0]:
# Step 16: Calculate Anomaly Scores


df["AnomalyScore"] = model.decision_function(X)

df[["AnomalyScore", "Prediction"]].head()

,AnomalyScore,Prediction
0,0.149795,1
1,0.105478,1
2,0.114638,1
3,0.138900,1
4,0.155716,1


In [0]:
# Step 17: Sort by Most Suspicious Events


df.sort_values(
    by="AnomalyScore",
    ascending=True
).head(10)

,User,Timestamp,Country,Hour,Status,Resource,Prediction,AnomalyScore
514,0,2026-07-09 00:01:00,2,0,0,4,-1,-0.132934
517,0,2026-07-15 00:59:00,3,0,0,4,-1,-0.130381
504,2,2026-07-07 02:27:00,1,2,0,4,-1,-0.127896
515,0,2026-07-10 01:57:00,2,1,0,0,-1,-0.124406
512,0,2026-07-29 03:28:00,1,3,0,2,-1,-0.122394
503,3,2026-07-28 01:36:00,2,1,0,4,-1,-0.106359
519,4,2026-07-08 00:22:00,3,0,0,3,-1,-0.102316
500,0,2026-07-03 04:08:00,2,4,0,1,-1,-0.101707
518,2,2026-07-05 03:00:00,3,3,0,4,-1,-0.100222
502,4,2026-07-07 03:51:00,4,3,0,0,-1,-0.094924


In [0]:
# Step 18: Create Human-Readable Dataset


# Create a fresh copy of the original login data
report_df = pd.DataFrame(rows)

# Copy ML results into the readable dataframe
report_df["Prediction"] = df["Prediction"]
report_df["AnomalyScore"] = df["AnomalyScore"]

print("Human-readable report created!")

report_df.head()

Human-readable report created!


,User,Timestamp,Country,Hour,Status,Resource,Prediction,AnomalyScore
0,David,2026-07-29 15:20:00,US,15,Success,Storage-1,1,0.149795
1,Charlie,2026-07-11 14:10:00,UK,14,Success,WebApp-1,1,0.105478
2,David,2026-07-24 15:02:00,US,15,Success,WebApp-1,1,0.114638
3,Bob,2026-07-12 15:29:00,US,15,Success,Storage-1,1,0.138900
4,David,2026-07-01 12:11:00,US,12,Success,Storage-1,1,0.155716


In [0]:
# Step 19: Display Human-Readable Anomalies


human_anomalies = report_df[report_df["Prediction"] == -1]

human_anomalies.sort_values(
    by="AnomalyScore",
    ascending=True
).head(10)

,User,Timestamp,Country,Hour,Status,Resource,Prediction,AnomalyScore
514,Alice,2026-07-09 00:01:00,Iran,0,Failed,WebApp-1,-1,-0.132934
517,Alice,2026-07-15 00:59:00,North Korea,0,Failed,WebApp-1,-1,-0.130381
504,Charlie,2026-07-07 02:27:00,China,2,Failed,WebApp-1,-1,-0.127896
515,Alice,2026-07-10 01:57:00,Iran,1,Failed,KeyVault-1,-1,-0.124406
512,Alice,2026-07-29 03:28:00,China,3,Failed,Storage-1,-1,-0.122394
503,David,2026-07-28 01:36:00,Iran,1,Failed,WebApp-1,-1,-0.106359
519,Eva,2026-07-08 00:22:00,North Korea,0,Failed,VM-1,-1,-0.102316
500,Alice,2026-07-03 04:08:00,Iran,4,Failed,SQL-DB-1,-1,-0.101707
518,Charlie,2026-07-05 03:00:00,North Korea,3,Failed,WebApp-1,-1,-0.100222
502,Eva,2026-07-07 03:51:00,Russia,3,Failed,KeyVault-1,-1,-0.094924


In [0]:
# Step 20: Install Azure OpenAI SDK


%pip install openai

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Step 21: Configure Azure OpenAI Client


from openai import AzureOpenAI

print("Azure OpenAI SDK Loaded Successfully!")

Azure OpenAI SDK Loaded Successfully!


In [0]:
# Step 22: Test Azure OpenAI Connection


from openai import AzureOpenAI

client = AzureOpenAI(
    api_key="YOUR_AZURE_OPENAI_API_KEY",
    api_version="2025-01-01-preview",
    azure_endpoint="https://poojaborade0707-9391-resource.services.ai.azure.com/"
)

deployment_name = "gpt-5-mini"

print("Connected!")

Connected!


In [0]:
# Step 23: Create AI Report Columns


report_df["AI_Explanation"] = ""
report_df["Severity"] = ""
report_df["Recommendation"] = ""
report_df["IncidentSummary"] = ""

print(report_df.columns)

Index(['User', 'Timestamp', 'Country', 'Hour', 'Status', 'Resource',
       'Prediction', 'AnomalyScore', 'AI_Explanation', 'Severity',
       'Recommendation', 'IncidentSummary'],
      dtype='object')


In [0]:
# Step 24: Select Top 5 Most Suspicious Anomalies

Sort anomalies by anomaly score (lowest = most suspicious)
top_anomalies = report_df[
    report_df["Prediction"] == -1
].sort_values(
    by="AnomalyScore",
    ascending=True
).head(5)

top_anomalies

,User,Timestamp,Country,Hour,Status,Resource,Prediction,AnomalyScore,AI_Explanation,Severity,Recommendation,IncidentSummary
514,Alice,2026-07-09 00:01:00,Iran,0,Failed,WebApp-1,-1,-0.132934,,,,
517,Alice,2026-07-15 00:59:00,North Korea,0,Failed,WebApp-1,-1,-0.130381,,,,
504,Charlie,2026-07-07 02:27:00,China,2,Failed,WebApp-1,-1,-0.127896,,,,
515,Alice,2026-07-10 01:57:00,Iran,1,Failed,KeyVault-1,-1,-0.124406,,,,
512,Alice,2026-07-29 03:28:00,China,3,Failed,Storage-1,-1,-0.122394,,,,


In [0]:
# Step 25: Generate and Save AI Security Analysis


for index, row in top_anomalies.iterrows():

    prompt = f"""
You are a cybersecurity analyst.

Analyze the following login event.

Provide your response in this exact format:

Explanation:
(2-3 sentences)

Severity:
(Low, Medium, High, or Critical)

Recommendation:
(3-5 bullet points)

Incident Summary:
(1 short paragraph)

Login Event

User: {row['User']}
Country: {row['Country']}
Hour: {row['Hour']}
Status: {row['Status']}
Resource: {row['Resource']}
"""

    response = client.chat.completions.create(
        model="gpt-5-mini",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    ai_response = response.choices[0].message.content

    print("=" * 80)
    print(f"Anomaly Index: {index}")
    print(ai_response)
    print("=" * 80)

Anomaly Index: 514
Explanation:
A failed login for Alice was observed from an IP geolocated to Iran at 00:00 targeting WebApp-1. While this single failed attempt has limited immediate impact, the unusual geography and timing are suspicious and could indicate reconnaissance or credential-stuffing activity.

Severity:
Medium

Recommendation:
- Review authentication logs (past 24–72 hours) for additional failed or successful attempts from the same IP, ASN, or geographic region and for other users hitting WebApp-1.
- Verify Alice’s normal login patterns (locations, devices) and contact her to confirm whether this was legitimate; if anomalous, require an immediate password reset and enforce MFA.
- Temporarily block or rate-limit the source IP/ASN and consider applying geoblocking or additional challenge requirements for traffic originating from high-risk countries.
- Enable/validate alerts for new-geo or out-of-hours successful logins and continue monitoring the account and WebApp-1 for any

In [0]:
# Step 26: Display Final AI Security Report


for index, row in top_anomalies.iterrows():

    prompt = f"""
You are a cybersecurity analyst.

Analyze the following login event.

Provide your response in this exact format:

Explanation:
(2-3 sentences)

Severity:
(Low, Medium, High, or Critical)

Recommendation:
(3-5 bullet points)

Incident Summary:
(1 short paragraph)

Login Event

User: {row['User']}
Country: {row['Country']}
Hour: {row['Hour']}
Status: {row['Status']}
Resource: {row['Resource']}
"""

    response = client.chat.completions.create(
        model="gpt-5-mini",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    ai_response = response.choices[0].message.content

    # Save the complete AI response
    report_df.loc[index, "AI_Explanation"] = ai_response

    print(f"Completed analysis for anomaly {index}")

Completed analysis for anomaly 514
Completed analysis for anomaly 517
Completed analysis for anomaly 504
Completed analysis for anomaly 515
Completed analysis for anomaly 512


In [0]:
# Step 27: Display Top 5 AI Investigation Results


report_df.loc[
    report_df["Prediction"] == -1,
    ["User", "Country", "AnomalyScore", "AI_Explanation"]
].head(5)

,User,Country,AnomalyScore,AI_Explanation
430,Eva,Canada,-0.015981,Explanation:\nA single successful authenticati...
500,Alice,Iran,-0.101707,
501,David,North Korea,-0.066190,
502,Eva,Russia,-0.094924,
503,David,Iran,-0.106359,


In [0]:
# Project Summary

In this project we:

- Generated simulated Azure login events
- Injected suspicious login attempts
- Used Isolation Forest to detect anomalies
- Ranked suspicious events
- Used Azure OpenAI GPT-5 Mini to explain each anomaly
- Produced an AI-assisted security investigation report